In [ ]:
!wget https://www.lamsade.dauphine.fr/~cazenave/project2025.zip
!unzip project2025.zip
!ls -l

In [ ]:
!pip install tensorrt-bindings==8.6.1
!pip install --extra-index-url https://pypi.nvidia.com tensorrt-libs
!pip install tensorflow[and-cuda]==2.15.0

# 1. Importing Required Libraries

In [ ]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import gc
import matplotlib.pyplot as plt
import golois
import json
import time
import os

from tensorflow.keras import layers, models, regularizers
from abc import ABC, abstractmethod

# 2. Define Constants

In [ ]:
planes = 31       # Number of input feature planes (each represents some Go board state feature)
moves = 361       # Total possible moves in a 19x19 Go board (19*19 = 361)
N = 10000         # Number of training samples
epochs = 20       # Number of training epochs
batch = 128       # Batch size for training
filters = 32      # Number of filters in CNN layers

# 3. Generate Random Training Data

In [ ]:
input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype('float32')

## 3.1 Move Probabilities

- Generates random move labels between 0 and 360.
- Uses one-hot encoding (keras.utils.to_categorical) so each move is represented as a vector of size 361, with 1 at the chosen move position.

In [ ]:
policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical(policy)

## 3.2 Value Labels

- Generates random win/loss labels (0 or 1), indicating whether the current player won.

In [ ]:
value = np.random.randint(2, size=(N,))
value = value.astype('float32')

## 3.3 End Game & Groups

In [ ]:
end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype('float32')

# 4. Printing Version & Validation Data

Calls golois.getValidation, a function from golois, for testing model inputs before training.

In [ ]:
print("Tensorflow version", tf.__version__)
print("getValidation", flush=True)
golois.getValidation(input_data, policy, value, end)

# 5. Building the Models. Example of a structure



# Plan

1. Have a structure that allows to easily handle / add several models
2. Select models you want for training
3. Automate training of selected models
4. Select & evaluate

In [ ]:
# Enable mixed precision training for speedup
try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
except:
    pass

# Directory paths
MODELS_DIR = "saved_models"
METADATA_FILE = os.path.join(MODELS_DIR, "metadata.json")

# Ensure directories exist
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

## 5.1 Have a structure that allows to easily handle / add several models

In [ ]:
# Dictionary to hold different model architectures
MODEL_ARCHITECTURES = {}

# Function to register models
def register_model(name):
    def wrapper(func):
        MODEL_ARCHITECTURES[name] = func
        return func
    return wrapper

# 1️⃣ Refactor Model Creation for Easy Expansion
@register_model("cnn_v1")
def create_cnn_v1(filters=32, planes=31):
    input_layer = layers.Input(shape=(19, 19, planes), name='board')
    x = layers.Conv2D(filters, 1, activation='relu', padding='same')(input_layer)
    for _ in range(5):
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    return create_policy_value_heads(x, input_layer)

@register_model("resnet_v1")
def create_resnet_v1(filters=64, planes=31):
    input_layer = layers.Input(shape=(19, 19, planes), name='board')
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(input_layer)
    for _ in range(5):
        x = residual_block(x, filters)
    return create_policy_value_heads(x, input_layer)

# Function to create policy and value heads
def create_policy_value_heads(x, input_layer):
    policy_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias=False)(x)
    policy_head = layers.Flatten()(policy_head)
    policy_head = layers.Activation('softmax', name='policy')(policy_head)

    value_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias=False)(x)
    value_head = layers.Flatten()(value_head)
    value_head = layers.Dense(50, activation='relu')(value_head)
    value_head = layers.Dense(1, activation='sigmoid', name='value')(value_head)

    model = models.Model(inputs=input_layer, outputs=[policy_head, value_head])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                  loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
                  metrics={'policy': 'categorical_accuracy', 'value': 'mse'})
    return model

# Residual Block for ResNet
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3,3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, (3,3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

# Retrieve a model dynamically
def get_model(model_name, **kwargs):
    if model_name in MODEL_ARCHITECTURES:
        return MODEL_ARCHITECTURES[model_name](**kwargs)
    else:
        raise ValueError(f"⚠️ Model '{model_name}' not found. Available models: {list(MODEL_ARCHITECTURES.keys())}")



## 5.2 Select models to train


In [ ]:
# model_names = ["cnn_v1", "resnet_v1"]  # List of models to train
model_names = ["resnet_v1"]

## 5.3 Training of models selected

In [ ]:
import os

if os.path.exists(METADATA_FILE):
    with open(METADATA_FILE, "r") as f:
        metadata = json.load(f)
else:
    metadata = {}

# Debug: Check if dataset is too large
print("Dataset Shape:", input_data.shape)
print("Policy Shape:", policy.shape)
print("Value Shape:", value.shape)

# Reduce dataset size for debugging (optional)
input_data = input_data[:1000]
policy = policy[:1000]
value = value[:1000]

# Optimize data loading
batch_size = 32
input_dataset = tf.data.Dataset.from_tensor_slices((input_data, {'policy': policy, 'value': value}))
dataset = input_dataset.batch(batch_size).prefetch(2)

# Check if dataset is loading correctly
for sample in dataset.take(1):
    print("Dataset loaded correctly!")

for model_name in model_names:
    print(f"🚀 Training model: {model_name}")
    model = get_model(model_name, filters=32)

    for epoch in range(1, 11):
        print(f'🔷 Epoch {epoch}')
        start_time = time.time()
        history = model.fit(dataset.take(1), epochs=1)  # Test on a small batch first
        print("Training started successfully!")
        training_time = time.time() - start_time

    # Save the trained model
    model_path = os.path.join(MODELS_DIR, f"{model_name}_final.keras")
    model.save(model_path)
    print(f"✅ {model_name} saved at {model_path}")

    # Store metadata
    metadata[model_name] = {
        "epochs": 10,
        "validation_loss": history.history['value_loss'][-1],
        "accuracy": history.history['policy_categorical_accuracy'][-1],
        "mse": history.history['value_mse'][-1],
        "training_time": round(training_time, 2),
        "timestamp": time.ctime()
    }

# Write metadata to JSON file
with open(METADATA_FILE, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"📄 Metadata updated.")


## 5.4 Evaluation & selection

In [ ]:
def get_best_model():
    if not os.path.exists(METADATA_FILE):
        print("⚠️ No metadata found. Train some models first.")
        return None

    with open(METADATA_FILE, "r") as f:
        metadata = json.load(f)

    if not metadata:
        print("⚠️ No models found in metadata.")
        return None

    best_model_name, best_model_data = max(metadata.items(), key=lambda x: x[1]["accuracy"])
    best_model_path = os.path.join(MODELS_DIR, f"{best_model_name}_final.keras")

    if not os.path.exists(best_model_path):
        print(f"⚠️ Best model '{best_model_name}' not found at {best_model_path}")
        return None

    print(f"🏆 Best Model: {best_model_name} | Accuracy: {best_model_data['accuracy']:.4f}")
    return tf.keras.models.load_model(best_model_path)

# Load the best model automatically
best_model = get_best_model()



In [ ]:
import matplotlib.pyplot as plt

def plot_model_performance(metadata_file):
    if not os.path.exists(metadata_file):
        print("⚠️ No metadata file found. Train some models first.")
        return

    with open(metadata_file, "r") as f:
        metadata = json.load(f)

    if not metadata:
        print("⚠️ No data found in metadata.")
        return

    plt.figure(figsize=(12, 5))

    for model_name, data in metadata.items():
        epochs = list(range(1, data["epochs"] + 1))
        accuracy = [data["accuracy"]] * len(epochs)
        validation_loss = [data["validation_loss"]] * len(epochs)
        mse = [data["mse"]] * len(epochs)

        # Plot Accuracy
        plt.subplot(1, 2, 1)
        plt.plot(epochs, accuracy, label=f"{model_name} Accuracy")
        plt.xlabel("Epochs")
        plt.ylabel("Accuracy")
        plt.title("Model Accuracy Over Epochs")
        plt.legend()

        # Plot Validation Loss
        plt.subplot(1, 2, 2)
        plt.plot(epochs, validation_loss, label=f"{model_name} Val Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Validation Loss")
        plt.title("Validation Loss Over Epochs")
        plt.legend()

    plt.show()

# Call the function to plot
plot_model_performance(METADATA_FILE)
